# 🌊 UNIFIED MAMBA EDGE MASTER SCREENING & PROGRESSION BENCHMARK
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doantrongthai/flood_mamba_edge/blob/main/master_mamba_screening_and_progression.ipynb)

### 15 Architectures | Seed 42 | Size 128x128 | Dual Dataset (FloodKaggle & AIFloodSense)

This notebook executes an end-to-end evaluation covering:
1. **LCMamNet Systematic Bottleneck Resolution Progression (Steps 0 → 4)**:
   - **Step 0**: `lcmamnet` (Baseline, 1.15M params, 5,440 tokens)
   - **Step 1**: `lcmam_step1` (Resolves Bottleneck 1: Decoupled Multi-Scale Tokens, 1,344 tokens, -75% tokens)
   - **Step 2**: `lcmam_step2` (Resolves Bottlenecks 1 + 2: + Asymmetric Strip Depthwise ASD-Conv 1x7 & 7x1)
   - **Step 3**: `lcmam_step3` (Resolves Bottlenecks 1 + 2 + 3: + Spatial-Gated Dilated Refinement SGR d=2)
   - **Step 4**: `aquamamnet` (Resolves Bottlenecks 1 + 2 + 3 + 4 simultaneously: + Lite-Boundary PixelShuffle Decoder + Mamba `expand=1`, 0.54M params)
2. **10 Novel Flood-Domain Edge Mamba Architectures**:
   - `flowmamba`, `wavemamba`, `morphomamba`, `ripplemamba`, `aeromamba`, `topomamba`, `hydromamba`, `deltamamba`, `boundarymamba`, `resmambanet`

All models run sequentially with **Seed 42** on **both datasets** at resolution **128x128**. Results are compiled into automated Leaderboard and Component Progression tables.

In [ ]:
# 1. Clone repository & Install base requirements
!git clone https://github.com/doantrongthai/flood_mamba_edge
%cd flood_mamba_edge
!pip install -r requirements.txt
!pip install -q tabulate fvcore thop pandas

In [ ]:
# 2. Setup Official Hardware-Accelerated Mamba CUDA Kernel on Google Colab GPU (Tesla T4)
import zipfile, json, sys, sysconfig, pathlib, subprocess, torch

SP = pathlib.Path(sysconfig.get_paths()["purelib"])
FILE_ID = "1kuBAKlNtIc8G1KQFqO8xV7t9ZEM4XATB"

assert torch.cuda.get_device_capability(0) == (7, 5), f"Vui long chon Colab GPU la Tesla T4 (capability 7.5), hien tai: {torch.cuda.get_device_name(0)}"

# 1) Tai goi pre-compiled CUDA wheel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown
gdown.download(id=FILE_ID, output="/content/pack.zip", quiet=False)

# 2) Giai nen vao site-packages
with zipfile.ZipFile("/content/pack.zip") as z:
    assert z.testzip() is None, "File tai ve bi hong, vui long tai lai"
    m = json.loads(z.read("__manifest__.json"))
    now = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda}
    bad = {k: (m[k], now[k]) for k in now if m[k] != now[k]}
    assert not bad, f"Moi truong Colab khong khop: {bad}"
    for name in z.namelist():
        if not name.startswith("__"):
            z.extract(name, SP)
print("Da giai nen vao:", SP)

# 3) Cai cac dependency bo tro
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "packaging", "ninja", "transformers", "triton"])

# 4) Kiem tra CUDA kernel Mamba tren GPU
import selective_scan_cuda, causal_conv1d_cuda
from mamba_ssm.modules.mamba_simple import causal_conv1d_fn
from mamba_ssm import Mamba
print("causal_conv1d_fn:", "OK (CUDA kernel)" if causal_conv1d_fn is not None else "FALLBACK")
y = Mamba(d_model=64).cuda()(torch.randn(2, 128, 64, device="cuda"))
y.mean().backward()
print("[OK] OFFICIAL MAMBA HARDWARE CUDA KERNEL ACTIVE:", y.shape)

In [ ]:
# 3. Download & Prepare Datasets (FloodKaggle & AIFloodSense)
import subprocess, sys

for ds in ["floodkaggle", "aifloodsense"]:
    print(f"Ensuring dataset '{ds}' is downloaded...")
    subprocess.run([sys.executable, "main.py", "--dataset", ds, "--download", "--epochs", "1", "--batch_size", "2", "--model", "unet", "--size", "128"], check=False)

print("✓ Datasets ready in data/ directory.")

In [ ]:
# 4. Master Architecture Registry & Configuration (Standard Defaults)
SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 100
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1
DATASETS = ["floodkaggle", "aifloodsense"]

# Group 1: Systematic Bottleneck Resolution Progression (Steps 0 -> 4)
PROGRESSION_MODELS = [
    "lcmamnet",     # Step 0: Baseline LCMamNet (1.15M params, 5,440 tokens)
    "lcmam_step1",  # Step 1: 1 Bottleneck Resolved (Decoupled Multi-Scale Tokens, 1,344 tokens, -75% tokens)
    "lcmam_step2",  # Step 2: 2 Bottlenecks Resolved (+ Asymmetric Strip Depthwise ASD-Conv 1x7 & 7x1)
    "lcmam_step3",  # Step 3: 3 Bottlenecks Resolved (+ Spatial-Gated Dilated Refinement SGR d=2)
    "aquamamnet",   # Step 4: All 4 Bottlenecks Resolved (+ Lite-Boundary PixelShuffle Decoder + expand=1, 0.54M params)
]

# Group 2: 10 Novel Flood-Domain Architectures
NOVEL_FLOOD_MODELS = [
    "flowmamba",      # Directional Strip Convolutions (1x7 & 7x1)
    "wavemamba",      # 2D Wavelet Haar Decomposition (High/Low frequency split)
    "morphomamba",    # Differentiable Morphological Gradient Gating (Dilation - Erosion)
    "ripplemamba",    # Concentric Multi-Stride Radial Scanning
    "aeromamba",      # Ultra-light UAV Drone Mamba (<0.1M params)
    "topomamba",      # Topographic Sobel Gradient-Modulated Mamba
    "hydromamba",     # Dual-Path Local CNN + Global Mamba Cross-Gating
    "deltamamba",     # Dilated Multi-Receptive Depthwise Convolutions (d=1,2,4)
    "boundarymamba",  # Internal High-Frequency Boundary Enhancement Prior
    "resmambanet",    # Inverted Residual Bottleneck with Mamba Token Mixer
]

# All 15 Models to Evaluate Sequentially
ALL_MODELS = PROGRESSION_MODELS + NOVEL_FLOOD_MODELS

print(f"Total architectures registered: {len(ALL_MODELS)}")
print(f"Progression Study: {len(PROGRESSION_MODELS)} models")
print(f"Novel Architectures: {len(NOVEL_FLOOD_MODELS)} models")
print(f"Default Parameters: Batch={BATCH_SIZE}, Size={IMAGE_SIZE}, Epochs={EPOCHS}, Loss={LOSS_NAME}, LR={LR}")

In [ ]:
# 5. Automated Hardware & Complexity Profiling Table
import pandas as pd
import torch
from models import get_model
from utils.metrics import calculate_model_complexity, measure_inference_time

device = "cuda" if torch.cuda.is_available() else "cpu"
profiling_records = []
print("=" * 90)
print(f"{'ARCH':<16} | {'CATEGORY':<14} | {'PARAMS':<10} | {'GFLOPS':<8} | {'LATENCY':<10} | {'FPS':<8}")
print("=" * 90)

for model_name in ALL_MODELS:
    try:
        model = get_model(model_name, num_classes=NUM_CLASSES, input_size=IMAGE_SIZE, seed=SEED).to(device)
        model.eval()
        complexity = calculate_model_complexity(model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device=device)
        perf = measure_inference_time(model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device=device)
        
        category = "Progression" if model_name in PROGRESSION_MODELS else "Novel Flood"
        profiling_records.append({
            "Architecture": model_name,
            "Category": category,
            "Params": complexity.get("deployable_params", complexity.get("total_params", complexity.get("params", 0))),
            "GFLOPs": complexity.get("gflops", 0.0),
            "Latency (ms)": perf.get("latency_ms", perf.get("latency_mean_ms", 0.0)),
            "FPS": perf.get("fps", 0.0)
        })
        p_val = profiling_records[-1]["Params"]
        g_val = profiling_records[-1]["GFLOPs"]
        lat_val = profiling_records[-1]["Latency (ms)"]
        fps_val = profiling_records[-1]["FPS"]
        print(f"{model_name:<16} | {category:<14} | {p_val:>10,d} | {g_val:>8.3f} | {lat_val:>8.2f} ms | {fps_val:>8.1f}")
        del model
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"{model_name:<16} | FAILED PROFILING: {e}")

df_profile = pd.DataFrame(profiling_records)
df_profile.to_csv("hardware_profiling_results.csv", index=False)
print("=" * 90)
print("✓ Hardware profiling completed and saved to hardware_profiling_results.csv")

In [ ]:
# 6. UNIFIED SEQUENTIAL SCREENING: Run all 15 models on FloodKaggle & AIFloodSense (Seed 42)
import re, subprocess, os, sys

screening_results = {}

for idx, model_name in enumerate(ALL_MODELS, 1):
    category = "Progression" if model_name in PROGRESSION_MODELS else "Novel Flood"
    print(f"\n{'#'*90}")
    print(f"[{idx}/{len(ALL_MODELS)}] BENCHMARKING: {model_name.upper()} ({category})")
    print(f"{'#'*90}")
    screening_results[model_name] = {}
    
    for ds in DATASETS:
        print(f"\n>>> Training {model_name} on {ds.upper()} (Seed {SEED}) <<<")
        cmd = [
            sys.executable, "main.py",
            "--dataset", ds,
            "--model", model_name,
            "--num_classes", str(NUM_CLASSES),
            "--loss", LOSS_NAME,
            "--size", str(IMAGE_SIZE),
            "--epochs", str(EPOCHS),
            "--batch_size", str(BATCH_SIZE),
            "--lr", str(LR),
            "--seed", str(SEED),
            "--download"
        ]
        
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        full_log = []
        for line in proc.stdout:
            full_log.append(line)
            if "Epoch [" in line or "mIoU:" in line or "Best" in line or "EVALUATION" in line or "IoU" in line:
                print(line, end="")
        proc.wait()
        
        log_str = "".join(full_log)
        
        # Parse metrics using regex
        miou = None
        dice = None
        loss = None
        
        miou_match = re.findall(r"mIoU[:=]\s*([0-9.]+)", log_str)
        if miou_match:
            miou = float(miou_match[-1])
            
        dice_match = re.findall(r"(?:Dice|F1)[:=]\s*([0-9.]+)", log_str)
        if dice_match:
            dice = float(dice_match[-1])
            
        loss_match = re.findall(r"(?:Val Loss|Loss)[:=]\s*([0-9.]+)", log_str)
        if loss_match:
            loss = float(loss_match[-1])
            
        screening_results[model_name][ds] = {
            "mIoU": miou,
            "Dice": dice,
            "Val_Loss": loss
        }
        print(f"--> [{model_name} on {ds}] Completed: mIoU={miou}, Dice={dice}, Val_Loss={loss}")

print("\n✓ All 15 training runs finished successfully!")

In [ ]:
# 7. SUMMARY LEADERBOARD & PROGRESSION STUDY TABLES
import pandas as pd
from tabulate import tabulate

master_rows = []
for m in ALL_MODELS:
    fk = screening_results.get(m, {}).get("floodkaggle", {})
    afs = screening_results.get(m, {}).get("aifloodsense", {})
    
    fk_miou = fk.get("mIoU") or 0.0
    fk_dice = fk.get("Dice") or 0.0
    afs_miou = afs.get("mIoU") or 0.0
    afs_dice = afs.get("Dice") or 0.0
    mean_miou = (fk_miou + afs_miou) / 2.0
    mean_dice = (fk_dice + afs_dice) / 2.0
    
    prof_row = df_profile[df_profile["Architecture"] == m]
    params = prof_row["Params"].values[0] if len(prof_row) > 0 else 0
    gflops = prof_row["GFLOPs"].values[0] if len(prof_row) > 0 else 0.0
    fps = prof_row["FPS"].values[0] if len(prof_row) > 0 else 0.0
    latency = prof_row["Latency (ms)"].values[0] if len(prof_row) > 0 else 0.0
    category = "Progression" if m in PROGRESSION_MODELS else "Novel Flood"
    
    master_rows.append({
        "Architecture": m,
        "Category": category,
        "Params": params,
        "GFLOPs": gflops,
        "FPS": fps,
        "Latency (ms)": latency,
        "FloodKaggle mIoU": fk_miou,
        "FloodKaggle Dice": fk_dice,
        "AIFloodSense mIoU": afs_miou,
        "AIFloodSense Dice": afs_dice,
        "Mean mIoU": mean_miou,
        "Mean Dice": mean_dice
    })

df_master = pd.DataFrame(master_rows)
df_master.to_csv("master_screening_results.csv", index=False)

# -------------------------------------------------------------
# TABLE 1: BOTTLENECK RESOLUTION PROGRESSION STUDY (Steps 0 -> 4)
# -------------------------------------------------------------
df_prog = df_master[df_master["Category"] == "Progression"].copy()
step_map = {
    "lcmamnet": "Step 0 (Baseline LCMamNet)",
    "lcmam_step1": "Step 1 (+ Decoupled Tokens)",
    "lcmam_step2": "Step 2 (+ ASD-Conv)",
    "lcmam_step3": "Step 3 (+ SGR)",
    "aquamamnet": "Step 4 (Proposed AquaMamNet: All 4 Resolved)"
}
df_prog["Step"] = df_prog["Architecture"].map(step_map)
cols_prog = ["Step", "Architecture", "Params", "GFLOPs", "FPS", "Mean mIoU", "Mean Dice"]
print("=" * 100)
print("TABLE 1: SYSTEMATIC BOTTLENECK RESOLUTION PROGRESSION STUDY")
print("=" * 100)
print(tabulate(df_prog[cols_prog], headers="keys", tablefmt="github", showindex=False))

# -------------------------------------------------------------
# TABLE 2: MASTER LEADERBOARD ACROSS ALL 15 ARCHITECTURES
# -------------------------------------------------------------
df_ranked = df_master.sort_values(by="Mean mIoU", ascending=False).reset_index(drop=True)
df_ranked.index += 1
cols_rank = ["Architecture", "Category", "Params", "GFLOPs", "FPS", "FloodKaggle mIoU", "AIFloodSense mIoU", "Mean mIoU", "Mean Dice"]
print("\n" + "=" * 100)
print("TABLE 2: MASTER LEADERBOARD ACROSS ALL 15 ARCHITECTURES (Ranked by Mean mIoU)")
print("=" * 100)
print(tabulate(df_ranked[cols_rank], headers="keys", tablefmt="github", showindex=True))

In [ ]:
# 8. Package Results & Download Archive
import shutil, os
from google.colab import files

os.makedirs("results_bundle", exist_ok=True)
for f in ["hardware_profiling_results.csv", "master_screening_results.csv"]:
    if os.path.exists(f):
        shutil.copy(f, os.path.join("results_bundle", f))

if os.path.exists("outputs"):
    for item in os.listdir("outputs"):
        p = os.path.join("outputs", item)
        if os.path.isfile(p) and item.endswith((".pth", ".csv", ".json")):
            shutil.copy(p, os.path.join("results_bundle", item))

shutil.make_archive("mamba_master_screening_results", "zip", "results_bundle")
print("✓ Results archived to mamba_master_screening_results.zip")
files.download("mamba_master_screening_results.zip")